# 지면 YOLO-seg 모델 학습_Colab

로컬에서 만든 `Train2YOLO_Outdoor_SurfaceGuideSeg.zip`을 Google Drive에 올린 뒤 Colab에서 학습합니다.

권장 Drive 구조:

```text
MyDrive/비전응용프로젝트/실외 보행 모델/지면/
├─ Train2YOLO_Outdoor_SurfaceGuideSeg.zip
└─ runs/                         # 자동 생성
```


In [3]:
# 필요 시 한 번만 실행
%pip install -U ultralytics pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.5 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import shutil
import yaml

from google.colab import drive
drive.mount('/content/drive')

DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
DRIVE_ROOT = DRIVE_MYDRIVE / '비전응용프로젝트/실외 보행 모델/2. 모델 생성/지면'
ZIP_NAME = 'Train2YOLO_Outdoor_SurfaceGuideSeg.zip'
DRIVE_ZIP = DRIVE_ROOT / ZIP_NAME
LOCAL_ROOT = Path('/content/Train2YOLO_Outdoor_SurfaceGuideSeg')
LOCAL_DATA_YAML = LOCAL_ROOT / 'data.yaml'
PROJECT_DIR = DRIVE_ROOT / 'runs'
RUN_NAME = 'outdoor_surface_guide_yolo26n-seg'

IMGSZ = 512
EPOCHS = 50   # 총 학습 epoch 목표. 현재 10 epoch 이후라면 50까지 이어 학습합니다.
BATCH = 16
WORKERS = 2
DEVICE = 0
RESUME = True   # 중단된 학습을 이어가려면 True. 새로 시작하려면 False.

# zip을 고정 경로에서 먼저 찾고, 없으면 MyDrive 전체에서 자동 검색합니다.
if not DRIVE_ZIP.exists():
    matches = sorted(DRIVE_MYDRIVE.rglob(ZIP_NAME))
    if matches:
        DRIVE_ZIP = matches[0]
        DRIVE_ROOT = DRIVE_ZIP.parent
        PROJECT_DIR = DRIVE_ROOT / 'runs'
        print('고정 경로에는 없지만 Drive에서 zip을 찾았습니다:', DRIVE_ZIP)
    else:
        print('zip을 찾지 못했습니다. Drive에 있는 zip 후보:')
        for p in sorted(DRIVE_MYDRIVE.rglob('*.zip'))[:50]:
            print('-', p)
        raise FileNotFoundError(f'{ZIP_NAME}을 MyDrive에서 찾지 못했습니다. 업로드 위치 또는 파일명을 확인하세요.')

print('DRIVE_ZIP:', DRIVE_ZIP, DRIVE_ZIP.exists())
print('LOCAL_ROOT:', LOCAL_ROOT)
print('PROJECT_DIR:', PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ZIP: /content/drive/MyDrive/비전응용프로젝트/실외 보행 모델/2. 모델 생성/지면/Train2YOLO_Outdoor_SurfaceGuideSeg.zip True
LOCAL_ROOT: /content/Train2YOLO_Outdoor_SurfaceGuideSeg
PROJECT_DIR: /content/drive/MyDrive/비전응용프로젝트/실외 보행 모델/2. 모델 생성/지면/runs


## 데이터셋 준비

In [5]:
RESET_LOCAL_DATASET = False  # 이전 압축해제 결과를 지우고 다시 풀려면 True

if RESET_LOCAL_DATASET and LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)

if not LOCAL_ROOT.exists():
    shutil.unpack_archive(str(DRIVE_ZIP), extract_dir='/content')

# zip 내부에 폴더가 포함되어 있지 않은 경우 보정
if not LOCAL_DATA_YAML.exists():
    candidates = list(Path('/content').glob('**/data.yaml'))
    print('data.yaml candidates:', candidates)
    if len(candidates) == 1:
        LOCAL_ROOT = candidates[0].parent
        LOCAL_DATA_YAML = candidates[0]
    else:
        raise FileNotFoundError('data.yaml을 찾지 못했습니다.')

# Colab 로컬 경로로 data.yaml patch
with open(LOCAL_DATA_YAML, 'r', encoding='utf-8') as f:
    data = yaml.safe_load(f)
data['path'] = str(LOCAL_ROOT)
with open(LOCAL_DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

print(LOCAL_DATA_YAML.read_text(encoding='utf-8'))
for rel in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    folder = LOCAL_ROOT / rel
    pattern = '*' if rel.startswith('images') else '*.txt'
    print(rel, len(list(folder.glob(pattern))) if folder.exists() else 'MISSING')


data.yaml candidates: [PosixPath('/content/data.yaml')]
path: /content
train: images/train
val: images/val
names:
  0: sidewalk
  1: braille_guide_blocks
  2: roadway
  3: alley
  4: crosswalk
  5: bike_lane
  6: caution_zone

images/train 41740
images/val 4656
labels/train 41740
labels/val 4656


## 학습 또는 이어서 학습

`RESUME=True`이면 Drive의 `runs/.../weights/last.pt`에서 이어서 학습합니다. `last.pt`가 없으면 오류를 내고 멈춥니다.


In [ ]:
from ultralytics import YOLO

run_dir = PROJECT_DIR / RUN_NAME
last_pt = run_dir / 'weights' / 'last.pt'
best_pt = run_dir / 'weights' / 'best.pt'

print('RUN_NAME:', RUN_NAME)
print('run_dir:', run_dir)
print('RESUME:', RESUME)
print('last.pt:', last_pt, last_pt.exists())
print('best.pt:', best_pt, best_pt.exists())

if RESUME:
    if not last_pt.exists():
        raise FileNotFoundError(
            f"이어 학습할 last.pt가 없습니다: {last_pt}\n"
            "Drive의 runs 폴더가 남아 있는지 확인하거나 RESUME=False로 새 학습을 시작하세요."
        )
    model = YOLO(str(last_pt))
    # resume=True는 last.pt의 학습 상태에서 이어가며, epochs는 최종 목표 epoch입니다.
    # 예: 이미 10 epoch까지 진행됐다면 EPOCHS=50일 때 50 epoch까지 이어갑니다.
    results = model.train(resume=True, epochs=EPOCHS)
else:
    model = YOLO('yolo26n-seg.pt')
    results = model.train(
        data=str(LOCAL_DATA_YAML),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        workers=WORKERS,
        device=DEVICE,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True,
        patience=25,
        cache=False,
    )


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int

## 결과 확인

In [ ]:
run_dir = PROJECT_DIR / RUN_NAME
for rel in ['weights/best.pt', 'weights/last.pt', 'results.csv', 'args.yaml']:
    p = run_dir / rel
    print(rel, p.exists(), p)

# 현재 몇 epoch까지 진행됐는지 간단히 확인합니다.
results_csv = run_dir / 'results.csv'
if results_csv.exists():
    lines = [line.strip() for line in results_csv.read_text(encoding='utf-8').splitlines() if line.strip()]
    if len(lines) >= 2:
        print('\n마지막 기록:')
        print(lines[-1])
        try:
            last_epoch = int(float(lines[-1].split(',')[0].strip()))
            print('마지막 완료 epoch:', last_epoch)
        except Exception:
            pass
